# Predicting the next trading day's direction

目标：在每个交易日收盘后，根据当天和过去的信息，预测下一交易日收盘时指数是否上涨。

## 1. Open the daily data

从本 notebook 所在的模块目录启动。每行是一个交易日，`Date` 是日期，`Close` 是收盘点位，`segment` 标明训练、验证或测试。

In [1]:
from pathlib import Path
import pandas as pd

feature_file = Path('../../data/processed/sp500_features.csv')
daily = pd.read_csv(feature_file, parse_dates=['Date']).sort_values('Date')
daily['segment'].value_counts()

segment
train    5083
val      1132
test     1132
Name: count, dtype: int64

## 2. Read three consecutive trading days

标签记在产生预测的那天。下一交易日收盘点位高于今天记为 `1`，低于或等于今天记为 `0`。先从表里手算，再用代码检查。

练习：9 月 3 日这行的答案是哪天知道的？9 月 4 日这行应该标成几？

In [2]:
example = daily.loc[daily['Date'].between('1997-09-03', '1997-09-05')]
example[['Date', 'Close', 'segment']].round({'Close': 2})

,Date,Close,segment
2,1997-09-03,927.86,train
3,1997-09-04,930.87,train
4,1997-09-05,929.05,train


## 3. Build the next-day label

数据已经按日期从早到晚排序。在每个 `segment`（训练、验证、测试）内部，把下一行的日期和收盘点位移到当前行。这样，同一行就能看到做预测的日期和核对答案的日期。

In [3]:
by_segment = daily.groupby('segment')
daily['target_date'] = by_segment['Date'].shift(-1)
daily['next_close'] = by_segment['Close'].shift(-1)

labeled = daily.dropna(subset=['next_close']).copy()
labeled['up'] = (labeled['next_close'] > labeled['Close']).astype(int)

`shift(-1)` 把下一行的值向上移一行。每段最后一个交易日在本段内没有下一行，所以先去掉这三个空答案，再比较点位。`up` 是模型要学的答案。训练时的输入只取预测当天已经知道的特征；`next_close` 和 `target_date` 仅供对齐和核对。

In [4]:
rows = labeled.loc[labeled['Date'].between('1997-09-03', '1997-09-05')]
rows[['Date', 'Close', 'target_date', 'next_close', 'up']].round({'Close': 2, 'next_close': 2})

,Date,Close,target_date,next_close,up
2,1997-09-03,927.86,1997-09-04,930.87,1
3,1997-09-04,930.87,1997-09-05,929.05,0
4,1997-09-05,929.05,1997-09-08,931.20,1


In [5]:
labeled['segment'].value_counts()

segment
train    5082
val      1131
test     1131
Name: count, dtype: int64

## 4. A simple baseline

先数一数训练集里涨和不涨各有多少天，选较常见的方向。接下来每个验证日都猜这个方向，用猜对的天数除以验证天数，得到准确率。

In [6]:
train = labeled.loc[labeled['segment'] == 'train']
val = labeled.loc[labeled['segment'] == 'val']

counts = train['up'].value_counts()
counts

up
1    2709
0    2373
Name: count, dtype: int64

In [7]:
baseline_guess = counts.idxmax()  # 出现次数最多的标签
correct = val['up'] == baseline_guess
print('Fixed prediction:', baseline_guess)
print('Correct predictions:', correct.sum(), '/', len(val))
print(f'Validation accuracy: {correct.mean():.2%}')

Fixed prediction: 1
Correct predictions: 625 / 1131
Validation accuracy: 55.26%


## 5. Logistic regression

先选三个收盘时已经知道的指标：上一交易日的对数收益率（log_ret_lag_1）、当前收盘点位相对 20 日均线的偏离（ma_ratio_20）、最近 21 个交易日的年化波动率（vol_21_ann）。先固定这些列，以便后面公平比较不同模型。

In [8]:
features = ['log_ret_lag_1', 'ma_ratio_20', 'vol_21_ann']
X_train = train[features]
y_train = train['up']
X_val = val[features]
X_train.head(3)

,log_ret_lag_1,ma_ratio_20,vol_21_ann
0,-0.011038,-0.029452,0.171044
1,-0.004659,0.002107,0.205698
2,0.030773,0.003739,0.205163


`X_train` 的每行是模型当天能看到的三项信息。`y_train` 是对应那行的真实答案。训练用这些历史记录学习；预测验证集时，只传入 `X_val`。

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# 标准化和模型参数都只用训练集估计。
logistic_model = make_pipeline(StandardScaler(), LogisticRegression())
logistic_model.fit(X_train, y_train)
val_prediction = logistic_model.predict(X_val)

In [10]:
logistic_correct = val_prediction == val['up']
print('Correct predictions:', logistic_correct.sum(), '/', len(val))
print(f'Logistic regression accuracy: {logistic_correct.mean():.2%}')
print(f'Majority baseline accuracy: {correct.mean():.2%}')

Correct predictions: 627 / 1131
Logistic regression accuracy: 55.44%
Majority baseline accuracy: 55.26%
